## 1. GDPR 법률 데이터와 사건 데이터 불러오기

In [5]:
import pandas as pd
import json

# 파일 경로
laws_path = "../HF_cache/KBs/GDPR/data-00000-of-00001.jsonl"
cases_path = "../HF_cache/cases/GDPR/balanced_sample.jsonl"

# JSONL 파일 로드 함수
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

# 데이터 불러오기
laws_data = load_jsonl(laws_path)
cases_data = load_jsonl(cases_path)

# DataFrame 변환
df_laws = pd.DataFrame(laws_data, columns=[
    "reference", "norm_type", "sender", "sender_role", "recipient", "recipient_role",
    "subject", "subject_role", "information_type", "consent_form", "purpose",
    "sender_is_subject", "recipient_is_subject", "regulation_id", "regulation_content"
])

df_cases = pd.DataFrame(cases_data, columns=[
    "norm_type", "sender", "sender_role", "recipient", "recipient_role",
    "subject", "subject_role", "information_type", "consent_form", "purpose",
    "followed_articles", "violated_articles", "case_content"
])

# 확인
print("=== GDPR 법률 데이터 ===")
display(df_laws.head())

print("\n=== GDPR 사건 데이터 ===")
display(df_cases.head())


=== GDPR 법률 데이터 ===


,reference,norm_type,sender,sender_role,recipient,recipient_role,subject,subject_role,information_type,consent_form,purpose,sender_is_subject,recipient_is_subject,regulation_id,regulation_content
0,{},"""General Definition""",[],[],[],[],[],[],[],null,[],"""Not Sure""","""Not Sure""",Article 1,"""Subject-matter and objectives"""
1,{},"""General Definition""",[],[],[],[],[],[],[],null,[],"""Not Sure""","""Not Sure""",Article 1(1),"""This Regulation lays down rules relating to t..."
2,{},"""General Definition""",[],[],[],[],[],[],[],null,[],"""Not Sure""","""Not Sure""",Article 1(2),"""This Regulation protects fundamental rights a..."
3,{},"""Permit""",[],[],[],[],[],[],[],null,[],"""Not Sure""","""Not Sure""",Article 1(3),"""The free movement of personal data within the..."
4,{},"""General Definition""",[],[],[],[],[],[],[],null,[],"""Not Sure""","""Not Sure""",Article 2,"""Material scope"""



=== GDPR 사건 데이터 ===


,norm_type,sender,sender_role,recipient,recipient_role,subject,subject_role,information_type,consent_form,purpose,followed_articles,violated_articles,case_content
0,permit,[Google],[Data Processor],[Google Data Centers],[Data Storage and Processing Entity],[Alex],[Data Subject],[Personal Information],Consent,Improving Services and Ensuring Security,"[Article 6 - Lawfulness of processing, Article...",[],"In 2023, a user named Alex from Germany discov..."
1,permit,[John Doe],[User],[Google Sites],[Service Provider],[John Doe],[User],[Email Address],Consent,Update Personal Information,"[Article 6 - Lawfulness of processing, Article...",[],"John Doe, a user of Google Sites, encountered ..."
2,permit,[John Doe],[User],[Google LLC],[Service Provider],[John Doe],[User],"[Personal Information, Address, Government-iss...",Authorization,Account Information Update,"[Article 5, Article 6, Article 32]",[],John Doe attempted to update his personal info...
3,permit,[GreenTech Supplies],[Business],[Amazon],[Service Provider],[GreenTech Supplies],[Business],[Email Content],None,Marketing,"[Article 6 - Lawfulness of processing, Article...",[],Amazon's automated systems flagged emails from...
4,permit,[Reddit],[Online Platform],[Law Enforcement Agency],[Law Enforcement],[Reddit Users],[Users],[User Information],Authorization,Investigation of potential violations of law,"[Article 6(1)(c) - Legal Obligation, Article 6...",[],Reddit received a request from a law enforceme...


## 2. Role, Attribute Graph 불러오기

In [6]:
# !pip install -q networkx lxml pandas

import networkx as nx
import pandas as pd
from pathlib import Path
from itertools import islice

# 파일 경로
ATTR_GRAPH_PATH = "/Users/taeyoonkwack/Documents/PrivaCI-Bench/updated_kgs/attribute_kg_88k.graphml"
ROLE_GRAPH_PATH = "/Users/taeyoonkwack/Documents/PrivaCI-Bench/updated_kgs/role_kg_45k.graphml"

# -------- 그래프 로드 --------
def load_graph(path: str) -> nx.DiGraph:
    G = nx.read_graphml(Path(path))
    if isinstance(G, (nx.MultiDiGraph, nx.MultiGraph)):
        H = nx.DiGraph()
        H.add_nodes_from(G.nodes(data=True))
        for u, v, data in G.edges(data=True):
            if not H.has_edge(u, v):
                H.add_edge(u, v, **data)
        G = H
    return G

# -------- 노드 DataFrame 변환 --------
def nodes_to_df(G: nx.DiGraph, limit: int | None = None) -> pd.DataFrame:
    rows = []
    iterator = G.nodes(data=True)
    if limit is not None:
        iterator = islice(iterator, limit)
    for n, attrs in iterator:
        row = {"node_id": n}
        row.update(attrs or {})
        rows.append(row)
    return pd.DataFrame(rows)

# -------- 엣지 DataFrame 변환 --------
def edges_to_df(G: nx.DiGraph, limit: int | None = None) -> pd.DataFrame:
    rows = []
    iterator = G.edges(data=True)
    if limit is not None:
        iterator = islice(iterator, limit)
    for u, v, data in iterator:
        d = dict(data or {})
        # label 또는 relation 키로 subsume / is subsumed by 들어있음
        relation = d.get("label") or d.get("relation") or ""
        # edge_source도 데이터에 들어있을 수 있음
        edge_source = d.get("source") or d.get("edge_source") or ""
        rows.append({
            "src_node": u,
            "dst_node": v,
            "edge_source": edge_source,
            "relation": relation
        })
    return pd.DataFrame(rows)

# -------- 그래프 불러오기 --------
G_attr = load_graph(ATTR_GRAPH_PATH)
G_role = load_graph(ROLE_GRAPH_PATH)

# -------- 노드/엣지 DataFrame 만들기 --------
df_attr_nodes = nodes_to_df(G_attr)
df_attr_edges = edges_to_df(G_attr)

df_role_nodes = nodes_to_df(G_role)
df_role_edges = edges_to_df(G_role)

# -------- 확인 --------
print("=== Attribute Graph: Nodes sample ===")
display(df_attr_nodes.head(5))

print("=== Attribute Graph: Edges sample ===")
display(df_attr_edges.head(5))

print("=== Role Graph: Nodes sample ===")
display(df_role_nodes.head(5))

print("=== Role Graph: Edges sample ===")
display(df_role_edges.head(5))

# relation 값 분포 확인
print("\nRelation values (Attribute Graph):", df_attr_edges["relation"].unique())
print("Relation values (Role Graph):", df_role_edges["relation"].unique())


=== Attribute Graph: Nodes sample ===


,node_id,domain
0,Financial Account,<https://w3id.org/dpv/dpv-owl/dpv-pd#Financial...
1,Account Identifier,<https://w3id.org/dpv/dpv-owl/dpv-pd#AccountId...
2,Behavioral,<https://w3id.org/dpv/dpv-owl/dpv-pd#Behavioral>
3,Vehicle Usage,<https://w3id.org/dpv/dpv-owl/dpv-pd#VehicleUs...
4,Identifying,<https://w3id.org/dpv/dpv-owl/dpv-pd#Identifying>


=== Attribute Graph: Edges sample ===


,src_node,dst_node,edge_source,relation
0,Financial Account,Account Identifier,GPT-4o,subsume
1,Financial Account,Payment Card,origin,subsume
2,Financial Account,Bank Account,GPT-4o,subsume
3,Financial Account,Financial,origin,is subsumed by
4,Financial Account,Savings Account,GPT-4o,subsume


=== Role Graph: Nodes sample ===


,node_id
0,person
1,inhabitant
2,female sibling
3,sister
4,evaluator


=== Role Graph: Edges sample ===


,src_node,dst_node,edge_source,relation
0,person,inhabitant,GPT-4o,is subsumed by
1,person,bad person,GPT-4o,subsume
2,person,female,GPT-4o,subsume
3,person,relative,GPT-4o,subsume
4,person,contestant,GPT-4o,subsume



Relation values (Attribute Graph): ['subsume' 'is subsumed by']
Relation values (Role Graph): ['is subsumed by' 'subsume']


In [ ]:
'''# !pip install -q networkx lxml pandas

import networkx as nx
import pandas as pd
from pathlib import Path
from itertools import islice

# 파일 경로
ATTR_GRAPH_PATH = "/Users/taeyoonkwack/Documents/PrivaCI-Bench/updated_kgs/attribute_kg_88k.graphml"

# ✅ ROLE_GRAPH 자리에 WordNet 그래프 사용
ROLE_GRAPH_PATH = "/Users/taeyoonkwack/Documents/PrivaCI-Bench/mycodes/0901_role_new.graphml"

# -------- 그래프 로드 --------
def load_graph(path: str) -> nx.DiGraph:
    G = nx.read_graphml(Path(path))
    # MultiGraph → DiGraph 변환 (첫 엣지만 보존)
    if isinstance(G, (nx.MultiDiGraph, nx.MultiGraph)):
        H = nx.DiGraph()
        H.add_nodes_from(G.nodes(data=True))
        for u, v, data in G.edges(data=True):
            if not H.has_edge(u, v):
                H.add_edge(u, v, **(data or {}))
        G = H
    else:
        # NetworkX가 GraphML을 DiGraph로 읽었더라도 혹시 모르니 타입 강제
        if not isinstance(G, nx.DiGraph):
            G = nx.DiGraph(G)
    return G

# -------- 노드 DataFrame 변환 --------
def nodes_to_df(G: nx.DiGraph, limit: int | None = None) -> pd.DataFrame:
    rows = []
    iterator = G.nodes(data=True)
    if limit is not None:
        iterator = islice(iterator, limit)
    for n, attrs in iterator:
        row = {"node_id": n}
        row.update(attrs or {})
        rows.append(row)
    return pd.DataFrame(rows)

# -------- 엣지 DataFrame 변환 --------
def edges_to_df(G: nx.DiGraph, limit: int | None = None,
                align_role_relation: bool = False) -> pd.DataFrame:
    """
    align_role_relation=True:
      - 원본 ROLE_GRAPH의 관계명에 맞춤.
      - WordNet graph의 relation이 'hypernym' / 'hyponym' 인 경우
        'is subsumed by'로 매핑 (특수 → 일반 방향은 'is subsumed by').
    """
    rows = []
    iterator = G.edges(data=True)
    if limit is not None:
        iterator = islice(iterator, limit)

    for u, v, data in iterator:
        d = dict(data or {})
        # 기존 키 우선 사용
        relation = d.get("label") or d.get("relation") or ""

        if align_role_relation:
            rel_l = str(relation).strip().lower()
            # WordNet 그래프에서 우리가 만든 엣지는 모두 특수 → 일반 방향
            # 따라서 ROLE_GRAPH 기준으로 'is subsumed by'로 통일
            if rel_l in {"hypernym", "hyponym"}:
                relation = "is subsumed by"

        edge_source = d.get("source") or d.get("edge_source") or ""
        rows.append({
            "src_node": u,
            "dst_node": v,
            "edge_source": edge_source,
            "relation": relation
        })
    return pd.DataFrame(rows)

# -------- 그래프 불러오기 --------
G_attr = load_graph(ATTR_GRAPH_PATH)
G_role = load_graph(ROLE_GRAPH_PATH)  # ← WordNet 그래프 대체

# -------- 노드/엣지 DataFrame 만들기 --------
df_attr_nodes = nodes_to_df(G_attr)
df_attr_edges = edges_to_df(G_attr)

# ✅ ROLE_GRAPH(DataFrame)는 관계명을 ROLE_GRAPH 규칙으로 정규화
df_role_nodes = nodes_to_df(G_role)
df_role_edges = edges_to_df(G_role, align_role_relation=True)

# -------- 확인 --------
print("=== Attribute Graph: Nodes sample ===")
display(df_attr_nodes.head(5))

print("=== Attribute Graph: Edges sample ===")
display(df_attr_edges.head(5))

print("=== Role Graph (WordNet 대체): Nodes sample ===")
display(df_role_nodes.head(5))

print("=== Role Graph (WordNet 대체): Edges sample ===")
display(df_role_edges.head(5))

# relation 값 분포 확인
print("\nRelation values (Attribute Graph):", df_attr_edges["relation"].unique())
print("Relation values (Role Graph/WordNet-aligned):", df_role_edges["relation"].unique())

# 노드/엣지 개수도 같이 출력
print(f"\n[Counts] Attribute Graph -> nodes: {G_attr.number_of_nodes()}, edges: {G_attr.number_of_edges()}")
print(f"[Counts] Role Graph (WordNet) -> nodes: {G_role.number_of_nodes()}, edges: {G_role.number_of_edges()}")
'''

=== Attribute Graph: Nodes sample ===


,node_id,domain
0,Financial Account,<https://w3id.org/dpv/dpv-owl/dpv-pd#Financial...
1,Account Identifier,<https://w3id.org/dpv/dpv-owl/dpv-pd#AccountId...
2,Behavioral,<https://w3id.org/dpv/dpv-owl/dpv-pd#Behavioral>
3,Vehicle Usage,<https://w3id.org/dpv/dpv-owl/dpv-pd#VehicleUs...
4,Identifying,<https://w3id.org/dpv/dpv-owl/dpv-pd#Identifying>


=== Attribute Graph: Edges sample ===


,src_node,dst_node,edge_source,relation
0,Financial Account,Account Identifier,GPT-4o,subsume
1,Financial Account,Payment Card,origin,subsume
2,Financial Account,Bank Account,GPT-4o,subsume
3,Financial Account,Financial,origin,is subsumed by
4,Financial Account,Savings Account,GPT-4o,subsume


=== Role Graph (WordNet 대체): Nodes sample ===


,node_id,origin,pos,in_wordnet
0,a,extracted,n,True
1,academic,extracted,n,True
2,accessed,extracted,n,False
3,accreditation,extracted,n,True
4,accredited,extracted,n,False


=== Role Graph (WordNet 대체): Edges sample ===


,src_node,dst_node,edge_source,relation
0,a,abstract entity,,is subsumed by
1,a,abstraction,,is subsumed by
2,a,aliment,,is subsumed by
3,a,alimentation,,is subsumed by
4,a,alkali,,is subsumed by



Relation values (Attribute Graph): ['subsume' 'is subsumed by']
Relation values (Role Graph/WordNet-aligned): ['is subsumed by']

[Counts] Attribute Graph -> nodes: 7875, edges: 176999
[Counts] Role Graph (WordNet) -> nodes: 9238, edges: 22739


## 3. 사건 데이터에 적용 가능한 법률 가져와보기 
### 3.1. adjacency list 구현 / 관련 함수 구현

In [7]:
import pandas as pd
import json
from collections import defaultdict, deque

# -----------------------------
# 그래프 유틸
# -----------------------------
def build_child_adj(df_edges: pd.DataFrame) -> dict[str, set[str]]:
    """
    relation ∈ {subsume, is subsumed by}.
    부모→자식(child) 인접리스트를 구축.
    """
    adj = defaultdict(set)
    for _, r in df_edges.iterrows():
        rel = str(r.get("relation") or "").strip()
        u = str(r.get("src_node"))
        v = str(r.get("dst_node"))
        if not u or not v:
            continue
        if rel == "subsume":
            parent, child = u, v
        elif rel == "is subsumed by":
            parent, child = v, u
        else:
            continue
        adj[parent].add(child)
    return adj

def descendants_or_self(anchor: str, child_adj: dict[str, set[str]]) -> set[str]:
    """anchor 포함, anchor에서 하위로 도달 가능한 모든 노드(자기 자신 포함)."""
    if not anchor:
        return set()
    seen = {anchor}
    q = deque([anchor])
    while q:
        cur = q.popleft()
        for nxt in child_adj.get(cur, ()):
            if nxt not in seen:
                seen.add(nxt)
                q.append(nxt)
    return seen

# 인접리스트 구성 (역할/속성 그래프 각각)
role_child_adj = build_child_adj(df_role_edges)
attr_child_adj = build_child_adj(df_attr_edges)

# -----------------------------
# 데이터 정규화
# -----------------------------
def to_list(x):
    """
    - list → 그대로
    - 문자열 "[]", "\"foo\"" 등은 JSON 파싱 시도
    - None/'null'/빈문자열 → []
    - 그 외 스칼라 → [str]
    """
    if x is None:
        return []
    if isinstance(x, list):
        return [str(t).strip() for t in x if str(t).strip()]
    if isinstance(x, str):
        s = x.strip()
        if not s or s.lower() == "null":
            return []
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")) or (s.startswith('"') and s.endswith('"')):
            try:
                j = json.loads(s)
                if isinstance(j, list):
                    return [str(t).strip() for t in j if str(t).strip()]
                if isinstance(j, str):
                    return [j.strip()] if j.strip() else []
            except Exception:
                pass
        return [s]
    return [str(x).strip()]

# -----------------------------
# 포함 판정 (요구사항 반영: 사건 ⊆ 법조항, 사건의 "모든" term이 커버되어야 함)
# -----------------------------
def all_case_terms_are_descendants_of_law_terms(
    law_terms: list[str],
    case_terms: list[str],
    child_adj: dict[str, set[str]],
    require_nonempty_case: bool = True,
) -> bool:
    """
    모든 '사건 term'이 법조항 term들 중 '어느 하나'의 하위/동일이어야 True.
    - law_terms가 비어 있으면 True (해당 필드 무조건 만족)
    - case_terms가 비어 있으면:
        require_nonempty_case=True  -> False
        require_nonempty_case=False -> True
    """
    if not law_terms:
        return True
    if not case_terms:
        return not require_nonempty_case

    # 법조항 각 term의 하위 포함 집합(자기 자신 포함)
    law_cover_sets = [descendants_or_self(lt, child_adj) for lt in law_terms]

    # 사건의 "모든" term이 커버되어야 함
    return all(any(ct in cov for cov in law_cover_sets) for ct in case_terms)

# -----------------------------
# 적용 가능 법조항 검색
# -----------------------------
def get_applicable_laws_for_case(
    case_row: pd.Series,
    df_laws: pd.DataFrame,
    role_child_adj: dict[str, set[str]],
    attr_child_adj: dict[str, set[str]],
    return_debug_cols: bool = True,
    only_permit: bool = False,
    exclude_recital: bool = False
) -> pd.DataFrame:

    case_sender    = to_list(case_row.get("sender_role"))
    case_recipient = to_list(case_row.get("recipient_role"))
    case_subject   = to_list(case_row.get("subject_role"))
    case_info_type = to_list(case_row.get("information_type"))

    rows = []
    for _, law in df_laws.iterrows():
        law_sender    = to_list(law.get("sender"))
        law_recipient = to_list(law.get("recipient"))
        law_subject   = to_list(law.get("subject"))
        law_info_type = to_list(law.get("information_type"))

        # ✅ 사건 ⊆ 법조항 (사건의 "모든" term이 법조항 term들의 하위/동일)
        cond_sender    = all_case_terms_are_descendants_of_law_terms(law_sender,    case_sender,    role_child_adj)
        cond_subject   = all_case_terms_are_descendants_of_law_terms(law_subject,   case_subject,   attr_child_adj)
        cond_recipient = all_case_terms_are_descendants_of_law_terms(law_recipient, case_recipient, role_child_adj)
        cond_info_type = all_case_terms_are_descendants_of_law_terms(law_info_type, case_info_type, attr_child_adj)

        if cond_sender and cond_subject and cond_recipient and cond_info_type:
            row = {
                "regulation_id": law.get("regulation_id"),
                "norm_type": law.get("norm_type"),
                "sender": law_sender,
                "recipient": law_recipient,
                "subject": law_subject,
                "information_type": law_info_type,
                "regulation_content": law.get("regulation_content"),
            }
            if return_debug_cols:
                row.update({
                    "_cond_sender(role)": cond_sender,
                    "_cond_subject(attr)": cond_subject,
                    "_cond_recipient(role)": cond_recipient,
                    "_cond_info_type(attr)": cond_info_type,
                })
            rows.append(row)

    df = pd.DataFrame(rows)

    # permit 필터링
    if only_permit and not df.empty:
        df = df[df["norm_type"].astype(str).str.strip('"').str.lower() == "permit"]

    # Recital 필터링
    if exclude_recital and not df.empty:
        df = df[~df["regulation_id"].astype(str).str.contains("Recital", case=False, na=False)]

    return df

# -----------------------------
# 시각화/실행 헬퍼
# -----------------------------
def visualize_retrieval(case_idx: int):
    case_row = df_cases.iloc[case_idx]

    print("=== 선택한 사건 요약 ===")
    display(pd.DataFrame({
        "norm_type": [case_row.get("norm_type")],
        "sender": [to_list(case_row.get("sender"))],
        "recipient": [to_list(case_row.get("recipient"))],
        "subject": [to_list(case_row.get("subject"))],
        "information_type": [to_list(case_row.get("information_type"))],
        "followed_articles": [case_row.get("followed_articles")],
        "violated_articles": [case_row.get("violated_articles")],
        "case_content": [case_row.get("case_content")[:300] + ("..." if case_row.get("case_content") and len(case_row.get("case_content")) > 300 else "")]
    }))

    # 사건의 followed_articles, violated_articles에 대응되는 법조항
    followed_articles = to_list(case_row.get("followed_articles"))
    violated_articles = to_list(case_row.get("violated_articles"))
    related_articles = set(followed_articles + violated_articles)
    df_related_laws = df_laws[df_laws["regulation_id"].isin(related_articles)]

    print("=== 사건에 명시적으로 연관된 법조항 (followed_articles + violated_articles) ===")
    if not df_related_laws.empty:
        display(df_related_laws[[
            "regulation_id", "norm_type", "sender", "recipient", "subject", "information_type", "regulation_content"
        ]])
    else:
        print("사건의 followed_articles, violated_articles에 해당하는 법조항이 없습니다.")

    # 그래프 기반 매칭 결과 (예: permit만, Recital 제외)
    df_applicable = get_applicable_laws_for_case(
        case_row=case_row,
        df_laws=df_laws,
        role_child_adj=role_child_adj,
        attr_child_adj=attr_child_adj,
        return_debug_cols=True,
        only_permit=True,
        exclude_recital=True,
    )

    print(f"=== 사건 #{case_idx} 에 적용 가능한 GDPR 법조항 (총 {len(df_applicable)}건) ===")
    if not df_applicable.empty:
        display(df_applicable[[
            "regulation_id", "norm_type", "sender", "recipient", "subject", "information_type", "regulation_content",
        ]])
    else:
        print("조건을 모두 만족하는 법조항이 없습니다.")


### 3.2. 사건 <- 적용가능 법규 가져오기

In [8]:
visualize_retrieval(120)  # 사건 번호를 입력하면 적용 가능한 법규를 가져오는 기능 필요시 바꾸세요 (논문의 예제가 50번)

=== 선택한 사건 요약 ===


,norm_type,sender,recipient,subject,information_type,followed_articles,violated_articles,case_content
0,prohibit,[Telekom Romania Mobile Communications S.A.],[Unauthorized individuals],"[99,210 data subjects, 413 customers]","[Customer number, Gender, Telephone number, Pe...",[],"[Art. 32 (1), Art. 32 (2)]",Telekom Romania Mobile Communications S.A. was...


=== 사건에 명시적으로 연관된 법조항 (followed_articles + violated_articles) ===
사건의 followed_articles, violated_articles에 해당하는 법조항이 없습니다.
=== 사건 #120 에 적용 가능한 GDPR 법조항 (총 49건) ===


,regulation_id,norm_type,sender,recipient,subject,information_type,regulation_content
3,Article 1(3),"""Permit""",[],[],[],[],"""The free movement of personal data within the..."
7,Article 2(4),"""Permit""",[],[],[],[],"""This Regulation shall be without prejudice to..."
38,Article 6(2),"""Permit""",[],[],[],[],"""Member States may maintain or introduce more ..."
45,Article 9(2),"""Permit""",[],[],[],[],"""Paragraph 1 shall not apply if one of the fol..."
49,Article 11(1),"""Permit""",[],[],[],[],"""If the purposes for which a controller proces..."
51,Article 12(2),"""Permit""",[],[],[],[],"""The controller shall facilitate the exercise ..."
52,Article 12(7),"""Permit""",[],[],[],[],"""The information to be provided to data subjec..."
62,Article 23(1),"""Permit""",[],[],[],[],"""Union or Member State law to which the data c..."
66,Article 24(2),"""Permit""",[Controller],[],[],[],"""Where proportionate in relation to processing..."
70,Article 25(3),"""Permit""",[],[],[],[],"""An approved certification mechanism pursuant ..."


## 4. GPT-4o-mini + GDPR 성능 측정
### 4.1. 필요한 함수 정의


In [9]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from sklearn.metrics import accuracy_score
from tqdm import tqdm   # ✅ tqdm 추가

# -----------------------------
# 환경 변수 로드 (OPENAI_API_KEY)
# -----------------------------
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# -----------------------------
# GPT 모델 초기화
# -----------------------------
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

# -----------------------------
# 프롬프트 템플릿 (출력 0/1 강제)
# -----------------------------
prompt_template = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a legal assistant. "
     "Given a case description and relevant GDPR articles, classify the case strictly as:\n"
     "- 0 if LEGAL\n"
     "- 1 if ILLEGAL\n"
     "Output only one character: 0 or 1."),
    ("human", 
     "Case:\n{case_content}\n\n"
     "Relevant GDPR Articles:\n{articles}\n\n"
     "Is this case legal (0) or illegal (1)?")
])

# -----------------------------
# 평가 함수 (tqdm 포함)
# -----------------------------
def evaluate_cases_binary(df_cases, df_laws, role_child_adj, attr_child_adj, K=10):
    preds, labels = [], []

    for idx in tqdm(range(min(K, len(df_cases))), desc="Evaluating cases", unit="case"):
        case_row = df_cases.iloc[idx]
        case_content = case_row.get("case_content", "")

        # 1. 적용 가능한 법률 가져오기
        df_applicable = get_applicable_laws_for_case(
            case_row=case_row,
            df_laws=df_laws,
            role_child_adj=role_child_adj,
            attr_child_adj=attr_child_adj,
            return_debug_cols=False,
            only_permit=False,
            exclude_recital=True,
        )

        # 2. regulation_id: regulation_content 문자열 만들기
        if not df_applicable.empty:
            articles = "\n".join(
                f"{row['regulation_id']}: {row['regulation_content']}"
                for _, row in df_applicable.iterrows()
            )
        else:
            articles = "None"

        # 3. GPT 호출
        chain = prompt_template | llm
        response = chain.invoke({
            "case_content": case_content,
            "articles": articles
        })
        gpt_output = response.content.strip()
        pred = "1" if gpt_output.startswith("1") else "0"  # 안전하게 파싱
        preds.append(int(pred))

        # 4. 정답 라벨 생성 (violated_articles 있으면 1, 없으면 0)
        violated_articles = to_list(case_row.get("violated_articles"))
        label = 1 if violated_articles else 0
        labels.append(label)

        # 중간 로그 출력
        #print(f"\n[Case {idx}] Pred: {pred}, Label: {label}")
        #print(f"Case Content: {case_content[:200]}...")
        #print(f"Applicable Articles: {articles[:200]}...\n")

    # 5. 정확도 계산
    acc = accuracy_score(labels, preds)
    print(f"=== Accuracy over {len(preds)} cases: {acc:.2%} ===")
    return preds, labels



### 4.2. 사건데이터 성능 측정해보기  

In [10]:
preds, labels = evaluate_cases_binary(
    df_cases=df_cases,
    df_laws=df_laws,
    role_child_adj=role_child_adj,
    attr_child_adj=attr_child_adj,
    K=1000
)

Evaluating cases: 100%|██████████| 150/150 [07:48<00:00,  3.12s/case]

=== Accuracy over 150 cases: 82.00% ===
